# Ophyd electrochemistry quickstart

This notebook runs the intended Ophyd/Bluesky workflow entirely against the deterministic simulator. It does not import PyVISA, discover instruments or communicate with hardware. It demonstrates immutable experiment configuration, the Flyer lifecycle, paired I/V events and a preliminary post-run visualization.

This is a living quickstart. Review it whenever the public request model, measurement/status schema, device lifecycle, target backend or commissioned capabilities change. The real 2460 adapter remains unavailable until the START/aperture timestamp relationship is established.

Run `uv sync --locked` from the repository root and use that environment as the notebook kernel. The first code cell locates the checkout and exposes its `src/` tree, so neither an editable package install nor a repository-root working directory is assumed.

In [1]:
import sys
from pathlib import Path

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "ophyd_electrochemistry").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from inside the ophyd-electrochemistry checkout")
source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

In [2]:
import threading
import time

from bluesky import RunEngine
from bluesky import plan_stubs as bps
from bluesky import preprocessors as bpp
from event_model import unpack_event_page

from ophyd_electrochemistry import (
    AcquisitionRequest,
    CurrentPulseSequence,
    DeviceState,
    StartMode,
    VoltagePulseSequence,
)
from ophyd_electrochemistry.keithley.k2460 import (
    DigitalIOConfig,
    Keithley2460Capabilities,
    Keithley2460Compiler,
    Keithley2460Config,
    Keithley2460Device,
    SafetyConfig,
    SourceCapabilities,
    TimingPolicy,
)
from ophyd_electrochemistry.simulation import FakeCell, FakeTransport, SimulatedRuntime

## 1. Define finite program examples and select one

The locally defined `config` and `capabilities` below are explicitly synthetic fixtures. They demonstrate the commissioned-configuration layer; they are not Keithley specifications or safe battery defaults. Three immutable examples are provided: unipolar current pulses, voltage pulses, and charge-balanced alternating positive/negative current dwells. Select one with `selected_program`. Short inline comments distinguish scientific settings, commissioned limits and infrastructure timeouts. Program-dependent host bounds are derived after compilation rather than copied as unrelated constants.

In [4]:
capabilities = Keithley2460Capabilities(
    profile_id="quickstart-synthetic-v1",
    evidence_kind="simulation",
    sources=(
        SourceCapabilities(
            source_function="current",
            level_min=-0.2,
            level_max=0.2,
            compliance_min=0.01,
            compliance_max=5.0,
            min_dwell_s=0.003,
            max_step=0.1,
            max_command_slope_per_s=10.0,
        ),
        SourceCapabilities(
            source_function="voltage",
            level_min=-5.0,
            level_max=5.0,
            compliance_min=0.001,
            compliance_max=0.2,
            min_dwell_s=0.003,
            max_step=1.0,
            max_command_slope_per_s=1000.0,
        ),
    ),
    timer_resolution_s=0.001,  # Quantization of all simulated schedules.
    source_action_time_s=0.001,
    source_settling_time_s=0.001,
    measurement_aperture_s=0.002,  # Integration window for one V/I sample.
    measurement_overhead_s=0.001,  # Dead time outside that aperture.
    min_measurement_period_s=0.003,  # Fastest commissioned sample cadence.
    max_duration_s=100.0,  # Longest accepted finite program.
    max_unique_points=2_000,
    max_logical_points=10_000,
    max_measurement_records=10_000,
    max_buffer_records=20_000,
    source_records_per_point=1,
    max_upload_bytes=2_000_000,
    max_trigger_blocks=5_000,
    fixed_trigger_blocks=12,
    trigger_blocks_per_point=2,
    max_tones=64,
    min_points_per_highest_tone=8,
    supported_off_modes=("simulation-off",),
    supported_terminals=("rear",),
    supported_sense_modes=("local",),
    supports_external_start=True,
    supports_external_abort=False,
    supports_local_cutoffs=True,
    local_cutoffs_use_measurements=True,
    supports_independent_measurement=True,
    aperture_may_cross_updates=True,
    abort_latency_s=0.01,
    cutoff_latency_s=0.01,
)
config = Keithley2460Config(
    visa_resource="SIM::NO_CONNECTION",  # Identifier only; no VISA is opened here.
    source_terminal="rear",  # Synthetic commissioned terminal selection.
    sense="local",  # Synthetic two-wire mode; hardware may commission remote.
    io=DigitalIOConfig(start_edge="rising"),  # Edge used only for external START.
    safety=SafetyConfig(
        voltage_min_v=-5.0,  # Absolute lower DUT safety envelope.
        voltage_max_v=5.0,  # Absolute upper DUT safety envelope.
        charge_current_max_a=0.2,  # Maximum allowed positive current.
        discharge_current_max_a=0.2,  # Maximum allowed current magnitude while sinking.
        power_abs_max_w=1.0,  # Maximum absolute commanded V×I bound.
        source_off_mode="simulation-off",  # Commissioned safe output-OFF behavior.
    ),
    timing=TimingPolicy(
        command_timeout_s=1.0,  # Bound for one host/backend transaction, not a run.
        external_start_timeout_s=10.0,  # Local wait for START before safe timeout.
        shutdown_timeout_s=1.0,  # Host wait for confirmed abort/output OFF.
        poll_period_s=0.1,  # Instrument-local START/timeout event polling cadence.
        required_abort_latency_s=0.02,  # Maximum acceptable commissioned abort latency.
        required_cutoff_latency_s=0.1,  # Maximum acceptable local protection latency.
    ),
)
current_pulse_program = CurrentPulseSequence(
    baseline_current_a=0.0,  # Current between pulses.
    pulse_current_a=0.01,  # Current during each pulse.
    pulse_width_s=0.020,  # ON portion of each period.
    period_s=0.050,  # Pulse-to-pulse interval; total duration is period × count.
    count=4,  # Number of complete periods.
    sample_period_s=0.010,  # Independent V/I aperture start cadence.
    voltage_limit_v=5.0,  # Compliance limit, not a voltage setpoint.
)
voltage_pulse_program = VoltagePulseSequence(
    baseline_voltage_v=0.10,  # Voltage between pulses.
    pulse_voltage_v=0.20,  # Voltage during each pulse.
    pulse_width_s=0.020,
    period_s=0.050,
    count=4,
    sample_period_s=0.010,
    current_limit_a=0.05,  # Compliance, not a current setpoint.
)
alternating_current_program = CurrentPulseSequence(
    pulse_current_a=0.01,  # Positive charge dwell.
    baseline_current_a=-0.01,  # Equal negative discharge dwell.
    pulse_width_s=0.25,  # Equal half-periods make commanded charge zero.
    period_s=0.50,
    count=4,
    sample_period_s=0.010,
    voltage_limit_v=5.0,
)
program_examples = {
    "current_pulse": current_pulse_program,
    "voltage_pulse": voltage_pulse_program,
    "alternating_current": alternating_current_program,
}
selected_program = "alternating_current"
program = program_examples[selected_program]
request = AcquisitionRequest(
    experiment_id=f"quickstart-synthetic-{selected_program}",
    start_mode=StartMode.IMMEDIATE,
    program=program,
)
compiled = Keithley2460Compiler(capabilities).compile(request, config)

# The host watchdog covers the compiled run, an optional local START wait, and one
# worst-case communication transaction. It is not a scientific user setting.
host_poll_period_s = 0.001
external_start_budget_s = (
    config.timing.external_start_timeout_s
    if request.start_mode == StartMode.EXTERNAL_TRIGGER
    else 0.0
)
host_completion_margin_s = max(config.timing.command_timeout_s, 2 * host_poll_period_s)
host_completion_timeout_s = (
    external_start_budget_s + compiled.achieved_duration_s + host_completion_margin_s
)
# The virtual instrument must advance beyond the compiled terminal tick. This has no
# relationship to wall-clock speed and is needed only by the offline simulator.
simulation_advance_ticks = compiled.source.duration_ticks + 1
{
    "hardware_ready": compiled.hardware_ready,
    "duration_s": compiled.achieved_duration_s,
    "measurement_records": compiled.budget.measurement_records,
    "host_completion_timeout_s": host_completion_timeout_s,
    "simulation_advance_ticks": simulation_advance_ticks,
    "request_sha256": compiled.request_sha256,
    "program_sha256": compiled.program_sha256,
}

{'hardware_ready': False,
 'duration_s': 2.0,
 'measurement_records': 200,
 'host_completion_timeout_s': 3.0,
 'simulation_advance_ticks': 2001,
 'request_sha256': '548ad061dae4d5a8967993731630c8bdd06a14074e37fbf8ea80ec4f367cde97',
 'program_sha256': '20687188ed8d321b8bf457d2490935e6476a7c915235e4ef9873823ff08d6ed7'}

## 2. Construct and run the device

`Keithley2460Device` consumes a typed backend. Here that backend is `FakeTransport` over `SimulatedRuntime`. The helper thread advances only the simulator's virtual clock; source timing remains runtime-owned rather than host-paced. Its tick budget comes from the compiled duration, so changing pulse width, period or count does not require editing a timeout elsewhere. The plan performs stage, prepare, kickoff, complete, collect and unstage.

In [5]:
runtime = SimulatedRuntime(
    capabilities=capabilities,
    config=config,
    cell=FakeCell(resistance_ohm=10.0, open_circuit_voltage_v=0.1),
)
backend = FakeTransport(runtime)
ec = Keithley2460Device(
    backend,
    name="ec",
    poll_period_s=host_poll_period_s,  # Host state polling; not sample cadence.
    completion_timeout_s=host_completion_timeout_s,  # Derived from this request.
    shutdown_timeout_s=config.timing.shutdown_timeout_s,
    collection_chunk_records=7,  # Retrieval batch size; does not change acquisition.
    # Voltage pulses are explicitly enabled only for this synthetic backend.
    supported_program_types=(CurrentPulseSequence, VoltagePulseSequence),
)

documents = []
run_engine = RunEngine({})
run_engine.subscribe(lambda name, document: documents.append((name, document)))
worker_errors = []


def finish_simulated_program():
    try:
        expected_state = (
            DeviceState.WAITING_START
            if request.start_mode == StartMode.EXTERNAL_TRIGGER
            else DeviceState.RUNNING
        )
        deadline = time.monotonic() + config.timing.command_timeout_s
        while ec.state != expected_state and time.monotonic() < deadline:
            time.sleep(0.001)
        if ec.state != expected_state:
            raise TimeoutError(f"Simulator did not enter {expected_state.value}")
        if expected_state == DeviceState.WAITING_START:
            runtime.set_inputs(start=True, abort=False)
        runtime.advance_ticks(simulation_advance_ticks)
    except BaseException as exc:
        worker_errors.append(exc)


def acquisition_plan():
    def cleanup():
        ec.stop(success=False)
        yield from bps.collect(ec, return_payload=False)

    @bpp.stage_decorator([ec])
    @bpp.run_decorator(md={"experiment_id": request.experiment_id, "role": "follower"})
    def run():
        def body():
            yield from bps.clear_checkpoint()
            yield from bps.prepare(ec, request, wait=True)
            yield from bps.kickoff(ec, wait=True)
            yield from bps.complete(ec, wait=True)
            yield from bps.collect(ec, return_payload=False)

        yield from bpp.finalize_wrapper(body(), cleanup())

    return (yield from run())


worker = threading.Thread(target=finish_simulated_program, daemon=True)
worker.start()
run_uids = run_engine(acquisition_plan())
worker.join(timeout=host_completion_timeout_s + config.timing.shutdown_timeout_s)
assert not worker.is_alive() and not worker_errors
assert ec.state == DeviceState.COMPLETE and ec.output_enabled is False
{"run_uids": run_uids, "final_state": ec.state.value, "output": ec.output_enabled}

{'run_uids': ('a687e15f-7867-4595-adc7-4312ba739511',),
 'final_state': 'complete',
 'output': False}

## 3. Inspect paired I/V records

The `electrochemistry` stream contains one event per complete paired sample. `ec_current` and `ec_voltage` therefore share the same aperture and mapping metadata. Source and measurement status remain separate untouched words. In this simulator, source-status bit 0 means synthetic compliance was active during at least part of the aperture; measurement status is currently zero. These are not Keithley bit definitions.

In [6]:
events = []
for name, document in documents:
    if name == "event":
        events.append(document)
    elif name == "event_page":
        events.extend(unpack_event_page(document))

assert events
rows = [event["data"] for event in events]
columns = (
    "ec_sample_index",
    "ec_time_relative",
    "ec_voltage",
    "ec_current",
    "ec_source_setpoint",
    "ec_mapping_quality",
    "ec_source_status",
    "ec_measurement_status",
)
print(" | ".join(columns))
print("-" * 118)
for row in rows[:12]:
    print(
        f"{row['ec_sample_index']:>3d} | "
        f"{row['ec_time_relative']:>8.4f} | "
        f"{row['ec_voltage']:>9.6f} | "
        f"{row['ec_current']:>9.6f} | "
        f"{row['ec_source_setpoint']:>9.6f} | "
        f"{row['ec_mapping_quality']:<22s} | "
        f"{row['ec_source_status']} | "
        f"{row['ec_measurement_status']}"
    )
print(f"\n{len(rows)} total paired records; table truncated to 12 rows.")

ec_sample_index | ec_time_relative | ec_voltage | ec_current | ec_source_setpoint | ec_mapping_quality | ec_source_status | ec_measurement_status
----------------------------------------------------------------------------------------------------------------------
  0 |   0.0020 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  1 |   0.0120 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  2 |   0.0220 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  3 |   0.0320 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  4 |   0.0420 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  5 |   0.0520 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  6 |   0.0620 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  7 |   0.0720 |  0.200000 |  0.010000 |  0.010000 | settled_single_dwell   | 0 | 0
  8 |   0.0820 |  0.200000 |  0.010000 |  0.010000 | settled_si

## 4. Preliminary visualization

The next cell creates an interactive Plotly figure with linked time axes and hover readouts. The command trace is placed on the axis matching the selected source function. Shaded regions mark the first dwell of each pulse period. This is a first-look diagnostic, not calibration, timing-acceptance or settling evidence.

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

times = [row["ec_time_relative"] for row in rows]
voltages = [row["ec_voltage"] for row in rows]
currents = [row["ec_current"] for row in rows]
setpoints = [row["ec_source_setpoint"] for row in rows]
source_function = compiled.source_function
commanded_row = 2 if source_function == "current" else 1
commanded_unit = "A" if source_function == "current" else "V"

figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Measured voltage", "Measured and commanded current"),
)
figure.add_trace(
    go.Scatter(
        x=times,
        y=voltages,
        mode="lines+markers",
        name="Voltage",
        hovertemplate="t=%{x:.6f} s<br>V=%{y:.6g} V<extra></extra>",
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=times,
        y=currents,
        mode="lines+markers",
        name="Measured current",
        hovertemplate="t=%{x:.6f} s<br>I=%{y:.6g} A<extra></extra>",
    ),
    row=2,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=times,
        y=setpoints,
        mode="lines",
        line={"dash": "dash"},
        name=f"Commanded {source_function}",
        hovertemplate=(
            f"t=%{{x:.6f}} s<br>{source_function.title()} set=%{{y:.6g}} "
            f"{commanded_unit}<extra></extra>"
        ),
    ),
    row=commanded_row,
    col=1,
)
for pulse_index in range(program.count):
    pulse_start = pulse_index * program.period_s
    pulse_end = pulse_start + program.pulse_width_s
    for row_number in (1, 2):
        figure.add_vrect(
            x0=pulse_start,
            x1=pulse_end,
            fillcolor="royalblue",
            opacity=0.08,
            line_width=0,
            row=row_number,
            col=1,
        )
figure.update_yaxes(title_text="Voltage (V)", row=1, col=1)
figure.update_yaxes(title_text="Current (A)", row=2, col=1)
figure.update_xaxes(title_text="Time from START (s)", row=2, col=1)
figure.update_layout(
    title="Simulated pulsed electrochemistry acquisition",
    template="plotly_white",
    hovermode="x unified",
    height=650,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.08},
)
figure.show()

## What changes for hardware?

The request, plan lifecycle, stream name, paired-record concept and separate `ec_source_status`/`ec_measurement_status` fields remain the same. A commissioned target backend replaces `FakeTransport`; startup owns the VISA resource, terminals/sense, digital I/O, safety envelope, timing requirements and buffer limits. The voltage-pulse example is enabled only by this notebook's explicit simulator allowlist and does not establish target support. Before any target backend is enabled, the timestamp proof must establish actual-START-relative aperture meaning and the adapter must supply the documented Keithley definitions for both status words.